# Sonda Note — Malayalam ASR fine-tune (Colab)

LoRA fine-tune of Whisper for Malayalam / Manglish. Runs on a **free T4**.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. Without this you get a CPU and nothing works.

### Read this first

Fine-tuning is **step 3**, not step 1. In order:

1. Build an eval set from your own recordings (free, and nobody else has one)
2. Measure your current pipeline against it
3. Fine-tune — *this notebook*
4. Compare. **Only scale up if the delta justifies it.**

Published scaling curves put the knee at **100–300 hours**. Fine-tuning on a narrow domain can make things *worse* — on 120h of maritime data, full fine-tuning degraded every Whisper size through overfitting. Measure, don't assume.

### Colab reality check

| | Free T4 | Colab Pro (L4/A100) |
|---|---|---|
| VRAM | 16GB | 24–40GB |
| bf16 | ❌ no (Turing) | ✅ yes |
| Session cap | ~12h, idle-kills at ~90min | longer, still capped |
| ~10h of audio, 3 epochs | ~4–6h | ~1–2h |

**Sessions die without warning.** Everything below checkpoints to Google Drive and auto-resumes, so a disconnect costs minutes, not the run.

## 1. Check the GPU

In [ ]:
import subprocess, sys

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")
print(out.stdout)

import torch
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Check compute capability, NOT torch.cuda.is_bf16_supported(): that helper
# counts EMULATED bf16 and so returns True on a T4, which has no bf16 hardware.
# Trusting it selects a config the card cannot run and OOMs.
major, minor = torch.cuda.get_device_capability(0)
bf16 = major >= 8   # sm_80 Ampere and newer

print(f"{name}  {vram:.0f}GB  compute {major}.{minor}  bf16(hardware)={bf16}")
if not bf16:
    print("-> pre-Ampere: use configs/whisper_ml_lora_colab.yaml (fp16). Correct here.")

## 2. Mount Drive

**Do not skip this.** Colab's local disk is wiped on disconnect. Checkpoints go to Drive so a dropped session is recoverable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
WORK = pathlib.Path('/content/drive/MyDrive/sondanote-asr')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'manifests').mkdir(exist_ok=True)
print('working dir:', WORK)

## 3. Install

Pinned versions. Colab's preinstalled `transformers` drifts and breaks the PEFT integration.

In [ ]:
%pip install -q transformers==4.48.0 datasets==3.2.0 peft==0.14.0 \
                accelerate==1.2.1 librosa==0.10.2.post1 soundfile==0.12.1 \
                evaluate==0.4.3 jiwer==3.0.5
print('done — if Colab asks you to restart the runtime, do it, then skip back to step 4')

## 4. Get the training code

Either clone your repo, or upload the `training/` folder to Drive.

In [ ]:
import pathlib, shutil, sys

# Option A — from Drive (upload training/ to MyDrive/sondanote-asr/training first)
src = WORK / 'training'

# Option B — from git:
# !git clone https://github.com/YOUR_ORG/meet-ai.git /content/meet-ai
# src = pathlib.Path('/content/meet-ai/training')

if not src.exists():
    raise SystemExit(f'Training code not found at {src}. Upload it or use the git option.')

TRAIN = pathlib.Path('/content/training')
if TRAIN.exists():
    shutil.rmtree(TRAIN)
shutil.copytree(src, TRAIN)
sys.path.insert(0, str(TRAIN / 'scripts'))
sys.path.insert(0, str(TRAIN / 'eval'))
print('code at', TRAIN)

## 5. Data

Manifests are JSONL, one utterance per line:

```json
{"audio": "/content/data/clip.wav", "text": "...", "duration": 4.2}
```

Generate yours locally with `scripts/build_evalset.py`, then upload both the manifest and the audio to Drive.

**Copy audio to local disk first.** Reading thousands of small files over the Drive FUSE mount is dramatically slower than local I/O and will dominate your training time.

In [ ]:
import json, pathlib, shutil, time

TRAIN_MANIFEST = WORK / 'manifests' / 'train.jsonl'
EVAL_MANIFEST  = WORK / 'manifests' / 'sondanote_eval.jsonl'

for path in (TRAIN_MANIFEST, EVAL_MANIFEST):
    if not path.exists():
        raise SystemExit(f'Missing {path} — upload your manifests to Drive first.')

def load(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

train, evalset = load(TRAIN_MANIFEST), load(EVAL_MANIFEST)
hours = sum(e.get('duration') or 0 for e in train) / 3600

print(f'train {len(train):>6} utterances  {hours:.2f}h')
print(f'eval  {len(evalset):>6} utterances')

if hours < 5:
    print('\nNOTE: under 5h. Gains begin around ~8h and are noisy below ~50h.')
    print('Worth running to validate the pipeline; do not expect a shippable model.')
elif hours > 50:
    est = hours * 3 * (49/800) * 2 * 3   # T4 is roughly 3x slower than a 4090
    print(f'\nEstimated T4 time: ~{est:.1f}h for 3 epochs.')
    if est > 11:
        print('That exceeds one Colab session (~12h). Auto-resume handles it,')
        print('but expect to reconnect and rerun the training cell 2-3 times.')

## 6. Baseline — measure BEFORE training

Skipping this is the most common mistake. Without a before-number, an after-number means nothing.

In [ ]:
import torch
from transformers import pipeline
from metrics import evaluate

BASE_MODEL = 'openai/whisper-medium'

asr = pipeline('automatic-speech-recognition', model=BASE_MODEL, device=0,
               generate_kwargs={'language': 'ml', 'task': 'transcribe'})

sample = evalset[:50]   # 50 is enough for a baseline; scoring is slow
pairs = []
for i, entry in enumerate(sample, 1):
    if not pathlib.Path(entry['audio']).exists():
        continue
    pairs.append((entry['text'], asr(entry['audio'])['text']))
    if i % 10 == 0:
        print(f'  {i}/{len(sample)}', flush=True)

baseline = evaluate(pairs, vocabulary=['Sonda Note', 'Supabase', 'Figma', 'Razorpay'])
print(f'\nBASELINE — {BASE_MODEL}\n')
print(baseline.summary())

del asr; torch.cuda.empty_cache()   # free VRAM before training

## 7. Train

`--auto-resume` continues from the newest Drive checkpoint. **If Colab disconnects, just rerun this cell.**

In [ ]:
# Point the config at the Drive manifests
import yaml

cfg_path = TRAIN / 'configs' / 'whisper_ml_lora_colab.yaml'
cfg = yaml.safe_load(open(cfg_path))
cfg['train_manifest'] = str(TRAIN_MANIFEST)
cfg['eval_manifest']  = str(EVAL_MANIFEST)
cfg['output_dir']     = str(WORK / 'whisper-ml-lora')

live = TRAIN / 'configs' / 'colab_live.yaml'
yaml.safe_dump(cfg, open(live, 'w'))
print(open(live).read())

In [ ]:
%cd /content/training
!python scripts/finetune.py --config configs/colab_live.yaml --dry-run

In [ ]:
!python scripts/finetune.py --config configs/colab_live.yaml --auto-resume

## 8. Did it help?

**Read CER first.** WER over-penalises Malayalam — it is agglutinative, so one wrong morpheme fails an entire long word. Human evaluation (NAACL 2025, Malayalam included) found CER tracks judgement better.

Also watch **entity recall**: whether your workspace vocabulary terms survived. That tracks product value far better than a 2-point WER move.

In [ ]:
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor

ADAPTER = str(WORK / 'whisper-ml-lora')

model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
model = PeftModel.from_pretrained(model, ADAPTER).merge_and_unload().to('cuda').eval()
processor = WhisperProcessor.from_pretrained(BASE_MODEL, language='ml', task='transcribe')

tuned = pipeline('automatic-speech-recognition', model=model,
                 tokenizer=processor.tokenizer, feature_extractor=processor.feature_extractor,
                 device=0, generate_kwargs={'language': 'ml', 'task': 'transcribe'})

pairs = []
for entry in sample:
    if pathlib.Path(entry['audio']).exists():
        pairs.append((entry['text'], tuned(entry['audio'])['text']))

after = evaluate(pairs, vocabulary=['Sonda Note', 'Supabase', 'Figma', 'Razorpay'])

print('BEFORE\n' + baseline.summary())
print('\nAFTER\n' + after.summary())

d_wer = after.wer.percent - baseline.wer.percent
d_cer = after.cer.percent - baseline.cer.percent
print(f'\nWER {d_wer:+.2f} points   CER {d_cer:+.2f} points   (negative = better)')

if d_cer < -2:
    print('\nReal improvement. Scaling to more data is justified.')
elif d_cer < 0:
    print('\nMarginal. More DATA will help more than more epochs.')
else:
    print('\nNo improvement — likely overfitting on too little data.')
    print('Do NOT scale up. Grow the dataset first, or lower lora_r / epochs.')

## 9. Export

The LoRA adapter is ~60MB. Download it, or serve it merged from your own box.

To use it in Sonda Note: add a provider to `apps/api/app/asr.py` implementing the `ASRProvider` protocol. Nothing else in the pipeline changes.

In [ ]:
import shutil

archive = shutil.make_archive(str(WORK / 'whisper-ml-lora-adapter'), 'zip', ADAPTER)
size_mb = pathlib.Path(archive).stat().st_size / 1024**2
print(f'{archive}  ({size_mb:.1f} MB)')
print('\nAlso saved in Drive, so it survives this session ending.')

# from google.colab import files; files.download(archive)